<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/nlp-labs/blob/main/Session2/1-Word-Embeddings-copy.ipynb)


# Maestría en Inteligencia Artificial  
## Procesamiento de Lenguaje natural
### Sesión 2 - Práctica

---


**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

# Introducción

# Clasificacion de textos - Analisis de reseñas y/u opiniones

En este notebook se aborda el problema de clasificación de texto aplicado al análisis de reseñas y opiniones en lenguaje natural. El objetivo principal es identificar automáticamente el sentimiento  presente en un conjunto de descripciones textuales utilizando técnicas de representación semántica.

# Configurar entorno

En esta sección se configuran las librerías y dependencias necesarias para el análisis de datos y procesamiento de lenguaje natural. Esto garantiza que el entorno esté listo para cargar, limpiar y analizar las conversaciones del chat político.

In [12]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

In [13]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [14]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
dataset = load_dataset('sepidmnorozy/Spanish_sentiment', split='train')
dataset

Dataset({
    features: ['label', 'text'],
    num_rows: 1029
})

In [15]:
dataset[1]

{'label': 1,
 'text': 'un lugar hermosisimo , inolvidable. hermosa atención · rescato el servicio , como asi también la calidez de su gente y el hermoso entorno natural. nunca olvidaremos este viaje. rescato el lugar , la playa. la variedad de comida y la calidad. desde bien tempranito al cruzarte con el personal siempre hay un saludo y buena onda. el tamaño de las habitaciones es espectacular. nos gusto mucho el spa , es un lugar muy familiar y para descansar'}

In [16]:
for i in range(5):
    print(dataset[i])

{'label': 1, 'text': 'Está situado en el centro de la ciudad , con todo lo más turístico a tu alrededor ( El Pilar , por ejemplo ) , y sitios para tomar algo .'}
{'label': 1, 'text': 'un lugar hermosisimo , inolvidable. hermosa atención · rescato el servicio , como asi también la calidez de su gente y el hermoso entorno natural. nunca olvidaremos este viaje. rescato el lugar , la playa. la variedad de comida y la calidad. desde bien tempranito al cruzarte con el personal siempre hay un saludo y buena onda. el tamaño de las habitaciones es espectacular. nos gusto mucho el spa , es un lugar muy familiar y para descansar'}
{'label': 1, 'text': 'Todo absolutamente recomendable .'}
{'label': 1, 'text': 'El metro está a 50 metros bajando tan solo una calle .'}
{'label': 1, 'text': 'Y de Toscana ... mejor que lo veais ... y además delo bonito que es Florencia y Siena ... perderos por sus pueblos ..'}


In [17]:
from collections import Counter
Counter(dataset["label"])

Counter({1: 851, 0: 178})

In [18]:
import numpy as np
np.mean([len(t.split()) for t in dataset["text"]])

np.float64(16.289601554907676)

In [19]:
text_lengths = [len(row['text']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

Texto más corto: 5
Texto más largo: 608
Longitud promedio: 84.85714285714286


In [20]:
import re
from collections import Counter

def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r"[^a-záéíóúüñ]+", " ", text)
    return text.strip().split()

# Construimos el vocabulario a partir de conjunto de datos.
token_counts = Counter()
for text in dataset["text"]:
    token_counts.update(simple_tokenizer(text))

# 50k-2 porque necesitamos reservar espacio para los dos tokens especiales
top_n_tokens = list(token_counts.keys())[:50000-2]
vocab = {"[PAD]": 0, "[UNK]": 1}
for token in top_n_tokens:
    vocab[token] = len(vocab)

def tokenize_text(text, max_length=50):
    tokens = simple_tokenizer(text)
    ids = [vocab.get(tok, vocab["[UNK]"]) for tok in tokens[:max_length]]
    ids += [vocab["[PAD]"]] * (max_length - len(ids))
    return ids

# 2. Preprocesamiento de datos

En esta etapa se realiza la limpieza y preparación del texto con el fin de garantizar que la información pueda ser procesada adecuadamente por los modelos de aprendizaje automático.

El preprocesamiento incluye la eliminación de símbolos innecesarios, caracteres especiales y signos de puntuación que no aportan valor semántico al análisis. Asimismo, se convierten todos los textos a minúsculas para evitar duplicidad de términos causada por diferencias en capitalización.

Este proceso permite estandarizar el corpus, reducir ruido en los datos y mejorar la calidad de las representaciones vectoriales generadas posteriormente.

In [21]:
print(f"Vocabulario: {len(vocab)} tokens")
print("Primeros 15 tokens:")
print(f"{top_n_tokens[:15]}")
print("15 tokens de en medio:")
print(f"{top_n_tokens[1000:1015]}")
print("Últimos 15 tokens:")
print(f"{top_n_tokens[-15:]}")

Vocabulario: 2672 tokens
Primeros 15 tokens:
['está', 'situado', 'en', 'el', 'centro', 'de', 'la', 'ciudad', 'con', 'todo', 'lo', 'más', 'turístico', 'a', 'tu']
15 tokens de en medio:
['valía', 'dam', 'ana', 'frank', 'min', 'caminando', 'central', 'sale', 'categoria', 'reflejar', 'casos', 'francamente', 'defraudaron', 'estaciones', 'autobuses']
Últimos 15 tokens:
['mariscal', 'victorio', 'lucchino', 'veía', 'divertí', 'cafeteria', 'malas', 'cuidan', 'vi', 'mercure', 'costo', 'sabes', 'doonde', 'obstante', 'quizás']
